# Module 1 - Foundations: Mapping the Terrain

**Time:** 09:30-11:00  
**Tool focus:** NetworkX

This notebook follows the first workshop block from the schedule PDF: loading standard graph formats, measuring baseline connectivity, extracting the network core, and visually inspecting segregation.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / ".cache"
(CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(context="talk", style="whitegrid")


In [ ]:
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph


def write_demo_graph_files(seed=42):
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    graph = build_demo_graph(seed=seed)

    node_frame = pd.DataFrame(
        [{"node_id": node, **attrs} for node, attrs in graph.nodes(data=True)]
    )
    edge_frame = pd.DataFrame(
        [{"source": source, "target": target} for source, target in graph.edges()]
    )

    nx.write_graphml(graph, DATA_RAW / "workshop_network.graphml")
    nx.write_gexf(graph, DATA_RAW / "workshop_network.gexf")
    nx.write_edgelist(graph, DATA_RAW / "workshop_network.edgelist", data=False)
    node_frame.to_csv(DATA_RAW / "workshop_nodes.csv", index=False)
    edge_frame.to_csv(DATA_PROCESSED / "workshop_edges.csv", index=False)


def load_graph(path):
    path = Path(path)
    if path.suffix == ".graphml":
        graph = nx.read_graphml(path)
    elif path.suffix == ".gexf":
        graph = nx.read_gexf(path)
    else:
        graph = nx.read_edgelist(path, nodetype=int)
        node_frame = pd.read_csv(DATA_RAW / "workshop_nodes.csv")
        attributes = node_frame.set_index("node_id").to_dict(orient="index")
        nx.set_node_attributes(graph, attributes)

    graph = nx.convert_node_labels_to_integers(graph, label_attribute="original_id")
    for _, attrs in graph.nodes(data=True):
        attrs["opinion"] = float(attrs["opinion"])
        attrs["activity"] = int(attrs["activity"])
        attrs["enclave"] = int(attrs["enclave"])
    return graph


## 1. Create or refresh the demo files

The schedule explicitly mentions GraphML, GEXF, and edge lists. We create all three so participants can see how the same network is represented in different formats.


In [ ]:
write_demo_graph_files()

paths = {
    "graphml": DATA_RAW / "workshop_network.graphml",
    "gexf": DATA_RAW / "workshop_network.gexf",
    "edgelist": DATA_RAW / "workshop_network.edgelist",
}
paths


In [ ]:
graphml_graph = load_graph(paths["graphml"])
gexf_graph = load_graph(paths["gexf"])
edgelist_graph = load_graph(paths["edgelist"])

pd.DataFrame(
    [
        {"format": "GraphML", "nodes": graphml_graph.number_of_nodes(), "edges": graphml_graph.number_of_edges()},
        {"format": "GEXF", "nodes": gexf_graph.number_of_nodes(), "edges": gexf_graph.number_of_edges()},
        {"format": "Edge list", "nodes": edgelist_graph.number_of_nodes(), "edges": edgelist_graph.number_of_edges()},
    ]
)


## 2. Establish a baseline for connectivity


In [ ]:
G = graphml_graph.copy()
largest_component = G.subgraph(max(nx.connected_components(G), key=len)).copy()

baseline = pd.Series(
    {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "density": nx.density(G),
        "average_degree": np.mean([degree for _, degree in G.degree()]),
        "average_clustering": nx.average_clustering(G),
        "average_path_length_lcc": nx.average_shortest_path_length(largest_component),
        "transitivity": nx.transitivity(G),
    }
).round(4)
baseline


In [ ]:
degree_stats = pd.DataFrame(
    [
        {
            "node_id": node,
            "degree": G.degree(node),
            "clustering": nx.clustering(G, node),
            "camp": G.nodes[node]["camp"],
            "community_label": G.nodes[node]["community_label"],
            "opinion": G.nodes[node]["opinion"],
        }
        for node in G.nodes()
    ]
).sort_values(["degree", "node_id"], ascending=[False, True])

degree_stats.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(degree_stats, x="degree", hue="camp", bins=15, ax=axes[0])
axes[0].set_title("Degree distribution")

sns.boxplot(data=degree_stats, x="camp", y="clustering", ax=axes[1])
axes[1].set_title("Local clustering by camp")

plt.tight_layout()


## 3. Separate the core from the periphery

We first inspect the degree distribution, then derive a `k` value from it. This keeps the filtering logic visible instead of treating the core extraction as a black box.


In [ ]:
degree_values = np.array([degree for _, degree in G.degree()])
k_threshold = max(2, int(np.quantile(degree_values, 0.7)))
core = nx.k_core(G, k=k_threshold)

pd.Series(
    {
        "k_threshold": k_threshold,
        "core_nodes": core.number_of_nodes(),
        "core_edges": core.number_of_edges(),
        "core_density": round(nx.density(core), 4),
    }
)


In [ ]:
pos = nx.spring_layout(G, seed=7)
camp_colors = ["#33658A" if G.nodes[node]["camp"] == "left" else "#D1495B" for node in G.nodes()]
core_nodes = set(core.nodes())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
nx.draw_networkx(G, pos=pos, node_color=camp_colors, node_size=50, edge_color="#BBBBBB", alpha=0.7, with_labels=False, ax=axes[0])
axes[0].set_title("Full network")

nx.draw_networkx(
    G,
    pos=pos,
    node_color=["#F4D35E" if node in core_nodes else "#DDDDDD" for node in G.nodes()],
    node_size=60,
    edge_color="#BBBBBB",
    alpha=0.7,
    with_labels=False,
    ax=axes[1],
)
axes[1].set_title("Core participants highlighted")
plt.tight_layout()
